# 🎮 LoRA Fine-tuning v2 (Game-optimized / KcELECTRA)

**보정된 UnSmile 데이터 + 수집된 게임 음성채팅 데이터**로 학습합니다.

## 📋 데이터 구성
- **UnSmile 보정 데이터**: 10,490건 (개인지칭 제외 + 48건 재라벨링)
- **게임 음성채팅 수집 데이터**: 519건 (긍정 285건 + 부정 234건)
- **총 Train 데이터**: 11,009건

## 📋 특징
- **모델**: `beomi/KcELECTRA-base-v2022` (한국어 최적화)
- **메트릭**: `abuse_recall` (악플 탐지율)
- **방식**: LoRA (Parameter-Efficient Fine-Tuning)

## 1. 환경 설정 및 라이브러리 임포트

In [ ]:
import os
import torch
import pandas as pd
import numpy as np
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from peft import LoraConfig, get_peft_model, TaskType
from sklearn.metrics import precision_recall_fscore_support, label_ranking_average_precision_score
from datasets import Dataset
import warnings
warnings.filterwarnings('ignore')

print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version: {torch.version.cuda}")

## 2. 하이퍼파라미터 설정

In [ ]:
MODEL_NAME = "beomi/KcELECTRA-base-v2022"
OUTPUT_DIR = "./output_v2_game_lora"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

EPOCHS = 10
BATCH_SIZE = 32
LEARNING_RATE = 2e-4
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01
MAX_LENGTH = 128

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.1

LABEL_NAMES = ["여성/가족", "남성", "성소수자", "인종/국적", "연령",
               "지역", "종교", "기타 혐오", "악플/욕설", "clean"]
NUM_LABELS = len(LABEL_NAMES)

print(f"Device: {DEVICE}")
print(f"Model: {MODEL_NAME}")

## 3. 데이터 로드 및 병합

**핵심**: UnSmile 보정 데이터 + 게임 음성채팅 수집 데이터 병합

In [ ]:
# 경로 설정 (Jupyter 환경에 맞게 수정)
UNSMILE_TRAIN = "../../3_UnSmile_Correction/unsmile_train_corrected.tsv"
UNSMILE_VALID = "../../3_UnSmile_Correction/unsmile_valid_corrected.tsv"
COLLECTED_DATA = "../../1_Data_Labeling_STT/keywords_unsmile_format.tsv"

# 데이터 로드
unsmile_train = pd.read_csv(UNSMILE_TRAIN, sep='\t', encoding='utf-8')
valid_df = pd.read_csv(UNSMILE_VALID, sep='\t', encoding='utf-8')
collected_df = pd.read_csv(COLLECTED_DATA, sep='\t', encoding='utf-8')

print(f"UnSmile Train: {len(unsmile_train)}건")
print(f"수집 데이터: {len(collected_df)}건")

# 데이터 병합
train_df = pd.concat([unsmile_train, collected_df], ignore_index=True)
print(f"\n✅ 병합된 Train 데이터: {len(train_df)}건")
print(f"Valid 데이터: {len(valid_df)}건")

# 라벨 분포 확인
print("\n📊 수집 데이터 라벨 분포:")
print(f"  - Clean (긍정): {collected_df['clean'].sum()}건")
print(f"  - 악플/욕설 (부정): {collected_df['악플/욕설'].sum()}건")

## 4. 토크나이저 및 전처리

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def preprocess_function(examples):
    tokenized = tokenizer(
        examples['문장'],
        padding='max_length',
        truncation=True,
        max_length=MAX_LENGTH
    )
    labels = [[float(examples[col][i]) for col in LABEL_NAMES] 
              for i in range(len(examples['문장']))]
    tokenized['labels'] = labels
    return tokenized

train_dataset = Dataset.from_pandas(train_df).map(
    preprocess_function, batched=True, remove_columns=train_df.columns.tolist())
valid_dataset = Dataset.from_pandas(valid_df).map(
    preprocess_function, batched=True, remove_columns=valid_df.columns.tolist())

print("전처리 완료!")

## 5. 모델 로드 및 LoRA 적용

In [ ]:
base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    problem_type="multi_label_classification"
)
base_model.config.id2label = {i: l for i, l in enumerate(LABEL_NAMES)}
base_model.config.label2id = {l: i for i, l in enumerate(LABEL_NAMES)}

peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=["query", "key", "value"],
    bias="none"
)

model = get_peft_model(base_model, peft_config).to(DEVICE)
model.print_trainable_parameters()

## 6. 평가 메트릭 정의

In [ ]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    probs = torch.sigmoid(torch.tensor(predictions)).numpy()
    preds = (probs > 0.5).astype(int)
    labels_int = labels.astype(int)
    
    abuse_p, abuse_r, abuse_f1, _ = precision_recall_fscore_support(
        labels_int[:,8], preds[:,8], average='binary', zero_division=0)
    clean_p, clean_r, clean_f1, _ = precision_recall_fscore_support(
        labels_int[:,9], preds[:,9], average='binary', zero_division=0)
    lrap = label_ranking_average_precision_score(labels, predictions)
    
    return {
        'lrap': lrap,
        'abuse_recall': abuse_r,
        'abuse_f1': abuse_f1,
        'clean_recall': clean_r,
        'clean_f1': clean_f1,
    }

## 7. 학습 설정 및 실행

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="abuse_recall",
    greater_is_better=True,
    logging_steps=50,
    save_total_limit=2,
    report_to="none",
    fp16=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

In [ ]:
print("🚀 학습 시작 (v2: 보정 데이터 + 수집 데이터)...")
trainer.train()
print("학습 완료!")

## 8. 모델 저장

In [ ]:
model.save_pretrained(f"{OUTPUT_DIR}/lora_adapter")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/lora_adapter")

merged_model = model.merge_and_unload()
merged_model.save_pretrained(f"{OUTPUT_DIR}/merged_model")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/merged_model")

print("모델 저장 완료!")

## 9. 최종 평가

In [ ]:
print("="*60)
print("📊 최종 평가 결과 (v2 LoRA Game-optimized)")
print("="*60)
print(f"📌 학습 데이터: UnSmile 보정({len(unsmile_train)}) + 수집({len(collected_df)}) = {len(train_df)}건")
print("="*60)

eval_results = trainer.evaluate()
for key, value in eval_results.items():
    print(f"  {key}: {value:.4f}")

with open(f"{OUTPUT_DIR}/results.txt", 'w', encoding='utf-8') as f:
    f.write("=== v2 LoRA Game-optimized (KcELECTRA) ===\n")
    f.write(f"Data: UnSmile corrected + Collected game chat\n")
    f.write(f"Train samples: {len(train_df)}\n")
    for k, v in eval_results.items():
        f.write(f"{k}: {v:.4f}\n")

print("="*60)
print("✅ v2 완료!")